# CFLN v9.0 â€” Kaggle Training Notebook

**Complex-domain Full-Layer Network** â€” 5-stage PSC-RPP-RL pipeline, 7 ablations,
5 OQ probes, three-domain continual-learning protocol.

| Setting | Value |
|---|---|
| Accelerator | GPU (T4 Ã-- 2 recommended) |
| Spec | `docs/CFLN_v900_Master_Spec.md` |
| Key invariants | `torch.cfloat` throughout; single `to_complex` / `to_real` |

**Sections**
1. Environment setup
2. Tokenizer setup (BPE + CTP special tokens)
3. Configurations (smoke / ablation / full)
4. Dataset helpers (WikiText-103 or synthetic)
5. Model + optimiser construction
6. Checkpoint save / load
7. Kaggle dataset persistence
8. Sanity checks
9. Training loop (OQ probes + auto-checkpoint)
10. Stage 0 â€” LM warmup
11. Stage 1 â€” PSC self-supervised training
12. Stage 2 â€” RPP-STaR trace generation
13. Stage 3 â€” SFT on RPP traces
14. Stage 4 â€” GRPO fine-tuning
15. Three-domain CL protocol (LANG â†’ CODE â†’ SCIENCE)
16. BWT evaluation
17. NeedleInHaystack tiered evaluation
18. Ablation series (A85â€“A93)
19. Generation demo (DCG+)

## 1. Environment setup

In [ ]:
import os, sys, subprocess

# â”€â”€ Clone repo if not already present â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
REPO_DIR = '/kaggle/working/cfln-lm'
if not os.path.isdir(REPO_DIR):
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ey3lock3r/cfln-lm.git',  # <-- set your repo URL
        REPO_DIR
    ], check=True)

# Install package (uv not available on Kaggle â€” use pip)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR,
                '--quiet', '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'wandb'],
               check=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready:', REPO_DIR)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import json, math, copy, time, gc, random
from pathlib import Path
import wandb

# â”€â”€ GPU / device â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPU  = torch.cuda.device_count()
print(f'Device: {DEVICE}  |  GPUs: {N_GPU}')
if DEVICE == 'cuda':
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory//2**30} GB')

# Deterministic seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Output dirs
WORK     = Path('/kaggle/working')
INPUT    = Path('/kaggle/input')
OUT_DIR  = WORK / 'cfln_output';    OUT_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUT_DIR / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
print('Output dir:', OUT_DIR)

## 2. Tokenizer setup

BPE tokenizer trained on WikiText-103 corpus. Must be built **once** and saved;
CTP special tokens (`<think>`, `</think>`) extended **before** model init so
`vocab_size` is correct at construction time.

In [ ]:
from tokenizers import ByteLevelBPETokenizer

VOCAB_SIZE = 8192  # matches CFG_ABLATION defaults
TOK_DIR    = WORK / 'tokenizer'

def build_tokenizer(corpus_path, vocab_size=VOCAB_SIZE, save_dir=TOK_DIR):
    """Build BPE tokenizer. Run ONCE, save, reload in later sessions."""
    save_dir = Path(save_dir); save_dir.mkdir(exist_ok=True)
    tok = ByteLevelBPETokenizer()
    tok.train(
        files=[str(corpus_path)],
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=['<pad>', '<unk>', '<s>', '</s>']
    )
    tok.save_model(str(save_dir))
    print(f'Tokenizer saved to {save_dir}. Vocab: {tok.get_vocab_size()}')
    return tok

def load_tokenizer(save_dir=TOK_DIR):
    tok = ByteLevelBPETokenizer(
        str(Path(save_dir) / 'vocab.json'),
        str(Path(save_dir) / 'merges.txt')
    )
    return tok

def extend_tokenizer_for_ctp(tok):
    """Add CTP + SSP special tokens. Must match model vocab_size."""
    tok.add_special_tokens(['<think>', '</think>', '<hypo>', '</hypo>',
                             '<push_goal>', '</push_goal>'])
    ids = {t: tok.token_to_id(t) for t in
           ['<think>', '</think>', '<hypo>', '</hypo>', '<push_goal>', '</push_goal>']}
    print('Special token IDs:', ids)
    return tok, ids

# â”€â”€ First session: build tokenizer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
WIKITEXT_TRAIN = INPUT / 'wikitext-103-raw-v1/wiki.train.raw'
TOKEN_IDS_PATH = WORK / 'token_ids.pt'

if TOK_DIR.exists() and (TOK_DIR / 'vocab.json').exists():
    print('Loading existing tokenizer...')
    tok = load_tokenizer()
    _, SPECIAL_IDS = extend_tokenizer_for_ctp(tok)
elif WIKITEXT_TRAIN.exists():
    print('Building tokenizer from WikiText-103...')
    tok = build_tokenizer(WIKITEXT_TRAIN)
    tok, SPECIAL_IDS = extend_tokenizer_for_ctp(tok)
    torch.save(SPECIAL_IDS, TOKEN_IDS_PATH)
else:
    print('No corpus found â€” using synthetic vocab (smoke only)')
    tok = None
    SPECIAL_IDS = {'<think>': VOCAB_SIZE, '</think>': VOCAB_SIZE + 1,
                   '<hypo>': VOCAB_SIZE + 2, '</hypo>': VOCAB_SIZE + 3,
                   '<push_goal>': VOCAB_SIZE + 4, '</push_goal>': VOCAB_SIZE + 5}

THINK_START_ID = SPECIAL_IDS['<think>']
THINK_END_ID   = SPECIAL_IDS['</think>']
ACTUAL_VOCAB   = (tok.get_vocab_size() if tok is not None else VOCAB_SIZE + 6)
print(f'Effective vocab_size: {ACTUAL_VOCAB}')

## 3. Configurations

| Name | Purpose | d_c | n_l | L |
|---|---|---|---|
| `SMOKE` | 3-step sanity check | 32 | 128 | 2 |
| `ABLATION` | Ablation runs | 64 | 320 | 3 |
| `FULL` | Production training | 256 | 2112 | 6 |

In [ ]:
from cfln.config import CFLNConfig
from dataclasses import asdict

def cfg_to_dict(cfg):
    return asdict(cfg)

def make_smoke_cfg(**overrides):
    """Tiny config â€” fits any GPU, runs in seconds."""
    cfg = CFLNConfig(
        d_c=32, n_l=128, n_p=16, L=2,
        vocab_size=ACTUAL_VOCAB, C_chunk=16,
        K_L1=8, K_L2=4, K_L3=4,
        d_ssm_fast=8, S_f=8,
        d_e_l=8, d_e_p=8,
        d_r_node=4, d_r_lista=16,
        k_l=8, k_l_min=4, k_l_max=8,
        n_heads_gat=2,
        K_hebb=4, K_sparse=8, N_rules=16,
        N_archive=16, N_iter=2, N_iter_refine=2,
        n_fourier=8, T_diff=10, D_g=4,
        T=32, B=2, stage=0,
    )
    for k, v in overrides.items():
        setattr(cfg, k, v)
    return cfg

def make_ablation_cfg(**overrides):
    """Medium config for ablation runs â€” ~30 min on T4."""
    cfg = CFLNConfig(
        d_c=64, n_l=320, n_p=32, L=3,
        vocab_size=ACTUAL_VOCAB, C_chunk=64,
        K_L1=32, K_L2=16, K_L3=16,
        d_r_node=4, d_r_lista=32,
        k_l=16, k_l_min=10, k_l_max=40,
        n_heads_gat=2,
        K_hebb=8, K_sparse=16, N_rules=256,
        N_archive=64, N_iter=2, N_iter_refine=4,
        n_fourier=16, T=128, B=4, stage=0,
    )
    for k, v in overrides.items():
        setattr(cfg, k, v)
    return cfg

def make_full_cfg(**overrides):
    """v9.0 spec defaults â€” full training run."""
    cfg = CFLNConfig(vocab_size=ACTUAL_VOCAB)
    for k, v in overrides.items():
        setattr(cfg, k, v)
    return cfg

SMOKE    = make_smoke_cfg()
ABLATION = make_ablation_cfg()
FULL     = make_full_cfg(gradient_checkpointing=True)

# Config for ablation PSC / GRPO stages
CFG_PSC  = make_ablation_cfg(stage=1)
CFG_GRPO = make_ablation_cfg(stage=4)

print('SMOKE    d_c=%d  n_l=%d  L=%d' % (SMOKE.d_c,    SMOKE.n_l,    SMOKE.L))
print('ABLATION d_c=%d  n_l=%d  L=%d' % (ABLATION.d_c, ABLATION.n_l, ABLATION.L))
print('FULL     d_c=%d  n_l=%d  L=%d' % (FULL.d_c,     FULL.n_l,     FULL.L))

## 4. Dataset helpers

**Phase 1 (smoke only):** TinyShakespeare â€” 488 steps/epoch at B=8,T=256. Overfit
after ~3 epochs. Use only for pipeline verification.

**Phase 2 (real experiments):** WikiText-103 â€” add via Kaggle UI `Add Data â†’
wikitext-103-raw-v1`. 103M train tokens.

In [ ]:
def tokenize_and_chunk(text_path, tok, chunk_size=256, stride=128):
    """Tokenize corpus â†’ overlapping chunks â†’ (N, T) int32 tensor."""
    text = open(text_path, encoding='utf-8').read()
    encoded = tok.encode(text)
    ids = torch.tensor(encoded.ids, dtype=torch.int32)
    chunks = []
    for i in range(0, len(ids) - chunk_size, stride):
        chunks.append(ids[i:i + chunk_size])
    data = torch.stack(chunks)
    print(f'Data: {len(data):,} chunks Ã-- {chunk_size} tokens = {len(data)*chunk_size/1e6:.1f}M tokens')
    return data

def sample_batch(data, batch_size, T, device=DEVICE):
    """Random batch of (B, T) long tensors."""
    idx = torch.randint(0, len(data), (batch_size,))
    return {'input_ids': data[idx].long().to(device)}

def make_synthetic_batches(n_batches, B, T, vocab_size, seed=0):
    """Generate random integer token batches for smoke tests."""
    rng = random.Random(seed)
    return [{'input_ids': torch.tensor(
        [[rng.randint(1, vocab_size - 1) for _ in range(T)] for _ in range(B)],
        dtype=torch.long)} for _ in range(n_batches)]

def load_domain_datasets(cfg, n_train=500, n_eval=50):
    """Load or synthesise domain datasets for CL protocol.
    Returns dict: {'LANG': [...], 'CODE': [...], 'SCIENCE': [...]}
    """
    V, B, T = cfg.vocab_size, cfg.B, cfg.T
    kaggle_paths = {
        'LANG':    str(INPUT / 'wikitext-103-raw-v1/wiki.train.raw'),
        'CODE':    str(INPUT / 'github-python/train.txt'),
        'SCIENCE': str(INPUT / 'pubmedqa/train.jsonl'),
    }
    datasets = {}
    for domain, path in kaggle_paths.items():
        if os.path.exists(path) and tok is not None:
            print(f'[{domain}] Loading from {path}')
            data = tokenize_and_chunk(path, tok, chunk_size=T)
            datasets[domain] = [sample_batch(data, B, T) for _ in range(n_train)]
        else:
            print(f'[{domain}] No dataset â€” using synthetic tokens')
            datasets[domain] = make_synthetic_batches(n_train, B, T, V, seed=hash(domain) % 9999)
    return datasets

print('Dataset helpers ready.')

## 5. Model + optimiser construction

In [ ]:
from cfln.modules.model import CFLNModel
from cfln.modules.si import SynapticIntelligence
from cfln.modules.monitoring import DocumentStreamingContext
from cfln.modules.psc_loss import PSCLoss
from cfln.training.optimizers import build_optimizers_v605


def build_model_and_opts(cfg, device=DEVICE):
    """Instantiate CFLNModel + SI + 5 optimisers + LR schedulers.

    Returns: model, opts (5-tuple), si, doc_ctx, psc_loss_fn, schedulers
    """
    cfg_dict = cfg_to_dict(cfg)

    model = CFLNModel(cfg_dict).to(device)
    model.train()

    si          = SynapticIntelligence(c_SI=cfg.c_SI)
    doc_ctx     = DocumentStreamingContext(model)
    psc_loss_fn = PSCLoss(
        d_c=cfg.d_c,
        d_r_lista=getattr(cfg, 'd_r_lista', None) or cfg.d_c // 2
    )

    opts = build_optimizers_v605(model, cfg_dict)
    # opts = (muon, muon_diff, opt_g, opt_u, opt_p)
    muon, muon_diff, opt_g, opt_u, opt_p = opts

    warmup = cfg_dict.get('warmup_steps', 500)
    schedulers = {
        'sched_g': torch.optim.lr_scheduler.LambdaLR(opt_g, lambda s: min(1.0, (s+1)/warmup)),
        'sched_u': torch.optim.lr_scheduler.LambdaLR(opt_u, lambda s: min(1.0, (s+1)/warmup)),
    }

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model: {n_params:,} trainable params  device={device}')
    return model, opts, si, doc_ctx, psc_loss_fn, schedulers


print('Builder ready.')

## 6. Checkpoint save / load

In [ ]:
def save_checkpoint(model, opts, si, schedulers, step, stage, path):
    muon, muon_diff, opt_g, opt_u, opt_p = opts
    ckpt = {
        'step': step, 'stage': stage,
        'model_state':    model.state_dict(),
        'muon_state':     muon.state_dict(),
        'muon_diff_state':muon_diff.state_dict(),
        'opt_g_state':    opt_g.state_dict(),
        'opt_u_state':    opt_u.state_dict(),
        'opt_p_state':    opt_p.state_dict(),
        'sched_g_state':  schedulers['sched_g'].state_dict(),
        'sched_u_state':  schedulers['sched_u'].state_dict(),
        # SI â€” NEVER reset between stages (CL memory)
        'si_omega':      {n: p.cpu().clone() for n, p in si.omega.items()},
        'si_theta_star': {n: p.cpu().clone() for n, p in si.theta_star.items()},
        # Non-param session state
        'titans_M':        model.encoder.titans.M.cpu().clone(),
        'bank_u_epi_mu':   torch.tensor(model.bank._u_epi_mu),
        'bank_u_epi_var':  torch.tensor(model.bank._u_epi_var),
        'bank_emin_mean':  torch.tensor(model.bank._Emin_mean),
        'bank_emin_var':   torch.tensor(model.bank._Emin_var),
        'bank_emin_n':     model.bank._Emin_n,
    }
    torch.save(ckpt, path)
    print(f'Saved: {Path(path).name}  (step={step}, stage={stage})')


def load_checkpoint(path, model, opts, si, schedulers, device=DEVICE):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    muon, muon_diff, opt_g, opt_u, opt_p = opts
    muon.load_state_dict(ckpt['muon_state'])
    muon_diff.load_state_dict(ckpt['muon_diff_state'])
    opt_g.load_state_dict(ckpt['opt_g_state'])
    opt_u.load_state_dict(ckpt['opt_u_state'])
    opt_p.load_state_dict(ckpt['opt_p_state'])
    schedulers['sched_g'].load_state_dict(ckpt['sched_g_state'])
    schedulers['sched_u'].load_state_dict(ckpt['sched_u_state'])
    for n, p in ckpt['si_omega'].items():      si.omega[n]      = p.to(device)
    for n, p in ckpt['si_theta_star'].items(): si.theta_star[n] = p.to(device)
    model.encoder.titans.M.copy_(ckpt['titans_M'].to(device))
    model.bank._u_epi_mu  = float(ckpt['bank_u_epi_mu'])
    model.bank._u_epi_var = float(ckpt['bank_u_epi_var'])
    model.bank._Emin_mean = float(ckpt['bank_emin_mean'])
    model.bank._Emin_var  = float(ckpt['bank_emin_var'])
    model.bank._Emin_n    = int(ckpt['bank_emin_n'])
    print(f'Loaded: step={ckpt["step"]}, stage={ckpt["stage"]}')
    return ckpt['step'], ckpt['stage']


print('Checkpoint helpers ready.')

## 7. Kaggle dataset persistence

In [ ]:
# â”€â”€ Setup Kaggle API credentials (first session, internet ON) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# In Kaggle: Add-ons â†’ Secrets â†’ add KAGGLE_KEY with your kaggle.json content
import os
os.makedirs('/root/.kaggle', exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret('KAGGLE_KEY')
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(secret)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle credentials configured.')
except Exception:
    print('No Kaggle secrets found â€” persistence via UI only.')


def upload_checkpoint(local_ckpt_path, dataset_title='cfln-checkpoints', step=0):
    """Upload checkpoint to Kaggle Dataset. Run before session ends."""
    import shutil
    upload_dir = WORK / 'upload'; upload_dir.mkdir(exist_ok=True)
    shutil.copy(local_ckpt_path, upload_dir)
    meta = {
        'title': dataset_title,
        'id': f'{os.environ.get("KAGGLE_USERNAME", "user")}/{dataset_title}',
        'licenses': [{'name': 'CC0-1.0'}]
    }
    with open(upload_dir / 'dataset-metadata.json', 'w') as f:
        json.dump(meta, f)
    os.system(f'kaggle datasets version -p {upload_dir} -m "step {step}"')
    print(f'Uploaded to dataset: {dataset_title}')


print('Persistence helpers ready.')

## 8. Sanity checks

Run before starting any training stage.

In [ ]:
from cfln.training.train_step import train_step_v605
from cfln.utils import verify_stiefel

print('=== SANITY CHECK ===')
sc_model, sc_opts, sc_si, sc_doc, _, sc_scheds = build_model_and_opts(SMOKE)
sc_batch = make_synthetic_batches(1, SMOKE.B, SMOKE.T, SMOKE.vocab_size)[0]
sc_batch = {k: v.to(DEVICE) for k, v in sc_batch.items()}

# 1. Forward pass
with torch.no_grad():
    logits, _, aux = sc_model(sc_batch['input_ids'], training=False)
assert logits.shape[0] == SMOKE.B and not torch.isnan(logits).any(), 'NaN in forward!'
print(f'  Forward: {logits.shape}  OK')

# 2. One training step
info = train_step_v605(
    sc_batch, sc_model, sc_opts, sc_si,
    phase='pretrain', step=0, total_steps=3,
    cfg=cfg_to_dict(SMOKE), doc_ctx=sc_doc,
)
loss_val = info.get('L_task', float('nan'))
assert not math.isnan(loss_val), f'NaN loss at sanity check: {loss_val}'
print(f'  Train step: loss={loss_val:.4f}  OK')

# 3. Stiefel constraint
assert verify_stiefel(sc_model.bank.W_l), 'Stiefel violated!'
print(f'  W_l Stiefel: OK')

# 4. Checkpoint round-trip
sc_ckpt = WORK / 'sanity_ckpt.pt'
save_checkpoint(sc_model, sc_opts, sc_si, sc_scheds, 0, 'sanity', sc_ckpt)
load_checkpoint(sc_ckpt, sc_model, sc_opts, sc_si, sc_scheds)
print(f'  Checkpoint round-trip: OK')

# 5. GPU memory
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    mem = torch.cuda.memory_allocated() / 1e9
    tot = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU used: {mem:.2f}GB / {tot:.1f}GB')

print('SANITY CHECK PASSED')
del sc_model, sc_opts, sc_si, sc_doc, sc_scheds
gc.collect()

## 9. Training loop

Shared loop with OQ probes, auto-checkpoint, and session-budget guard.

In [ ]:
SESSION_BUDGET_H = 8.5   # save and stop before Kaggle kills session
CKPT_EVERY       = 500
LOG_EVERY        = 100


class OQProbes:
    """Instruments for Open Questions (OQ-v598-2, OQ-v598-4, OQ-v596-3)."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.escape_count = 0
        self.rule_writes  = 0
        self.rule_reads   = 0
        self.domain_shifts = 0
        self.u_epi_vals   = []
        self.ce_vals      = []
        self.n_active_units = []
        self._last_domain = None

    def log(self, info, loss):
        self.u_epi_vals.append(info.get('U_epistemic', 0.5))
        self.ce_vals.append(float(loss))
        self.n_active_units.append(info.get('n_sensory', 0))
        # escape_fired / rule_write / rule_read not emitted by train_step
        _cur_domain = info.get('domain', None)
        if _cur_domain is not None and _cur_domain != self._last_domain:
            self.domain_shifts += 1
        self._last_domain = _cur_domain

    def summary(self, window=500):
        import statistics
        n = min(window, len(self.u_epi_vals))
        if n < 10:
            return {}
        u  = self.u_epi_vals[-n:]
        ce = self.ce_vals[-n:]
        try:
            corr = (sum((a - statistics.mean(u)) * (b - statistics.mean(ce))
                        for a, b in zip(u, ce)) /
                    (n * statistics.stdev(u) * statistics.stdev(ce) + 1e-8))
        except Exception:
            corr = 0.0
        return {
            'u_epi_ce_corr':   corr,                                  # OQ-v598-2 target >0.4
            'escape_rate':     self.escape_count / max(n, 1),
            'rule_write_rate': self.rule_writes  / max(n, 1),
            'rule_read_rate':  self.rule_reads   / max(n, 1),
            'domain_shifts':   self.domain_shifts,
            'n_active_units':  self.n_active_units[-1] if self.n_active_units else 0,
        }


def training_loop(model, opts, si, schedulers, data_or_fn, cfg,
                  stage='stage0', start_step=0, n_steps=None,
                  train_fn=None, doc_ctx=None, wandb_run=None):
    """
    Generic training loop.
    data_or_fn: list of batch dicts OR callable(step) -> batch dict
    train_fn: callable(batch, step) -> info dict; defaults to train_step_v605
    """
    cfg_dict  = cfg_to_dict(cfg) if not isinstance(cfg, dict) else cfg
    n_steps   = n_steps or cfg_dict.get('n_steps', 10_000)
    muon, muon_diff, opt_g, opt_u, opt_p = opts

    if train_fn is None:
        train_fn = lambda batch, step: train_step_v605(
            batch, model, opts, si,
            phase='pretrain', step=step,
            total_steps=n_steps, cfg=cfg_dict, doc_ctx=doc_ctx)

    probes   = OQProbes()
    loss_hist = []
    t0 = time.time()

    for step in range(start_step, start_step + n_steps):
        elapsed_h = (time.time() - t0) / 3600
        if elapsed_h >= SESSION_BUDGET_H:
            path = CKPT_DIR / f'ckpt_{stage}_{step:06d}_autosave.pt'
            save_checkpoint(model, opts, si, schedulers, step, stage, path)
            print(f'Session budget reached at step {step}. Saved and stopping.')
            break

        if callable(data_or_fn):
            batch = data_or_fn(step)
        else:
            batch = data_or_fn[step % len(data_or_fn)]
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        info = train_fn(batch, step)
        schedulers['sched_g'].step()
        schedulers['sched_u'].step()

        loss_val = info.get('L_task', float('nan'))
        probes.log(info, loss_val)
        loss_hist.append(float(loss_val))

        if step % LOG_EVERY == 0:
            avg = sum(loss_hist[-LOG_EVERY:]) / min(len(loss_hist), LOG_EVERY)
            oq  = probes.summary()
            print(f'[{stage}] {step:6d} | loss={avg:.4f} | '
                  f'U_epi={info.get("U_epistemic",0):.3f} | '
                  f'escape={oq.get("escape_rate",0):.3f} | '
                  f'rule_w={oq.get("rule_write_rate",0):.3f} | '
                  f'n_l={oq.get("n_active_units",0)} | '
                  f't={elapsed_h:.2f}h')
            if wandb_run:
                wandb.log({'loss': avg, 'stage': stage, 'step': step, **oq})

        if step % CKPT_EVERY == 0 and step > start_step:
            path = CKPT_DIR / f'ckpt_{stage}_{step:06d}.pt'
            save_checkpoint(model, opts, si, schedulers, step, stage, path)

    return loss_hist, probes


@torch.no_grad()
def evaluate(model, val_batches, cfg, n_batches=100):
    """Perplexity on validation batches. Freezes unit growth during eval."""
    cfg_dict = cfg_to_dict(cfg) if not isinstance(cfg, dict) else cfg
    model.eval()
    model.reset_for_inference()
    original_growth = getattr(model.bank, '_allow_growth', True)
    model.bank._allow_growth = False
    total_ce = 0.0
    count = 0
    for batch in val_batches[:n_batches]:
        ids = batch['input_ids'].to(DEVICE)
        logits, _, _ = model(ids, training=False)
        targets = ids[:, 1:]
        if logits.shape[1] > targets.shape[1]:
            logits = logits[:, :targets.shape[1]]
        ce = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=0,
        )
        total_ce += float(ce)
        count += 1
    model.bank._allow_growth = original_growth
    model.train()
    mean_ce = total_ce / max(count, 1)
    return {'val_ce': mean_ce, 'val_ppl': math.exp(mean_ce)}


print('Training loop ready.')

## 10. Stage 0 â€” LM warmup

Standard language model pre-training. Establishes CNEP routing and basic representations.
Expected time: ~2h on T4 for 10K steps at ablation scale.

In [ ]:
RUN_CFG = ABLATION  # change to FULL for production

N_PRETRAIN_STEPS = 200   # set to ~10000 for a real run

print(f'=== STAGE 0 PRE-TRAINING  ({N_PRETRAIN_STEPS} steps) ===')
model, opts, si, doc_ctx, psc_loss_fn, schedulers = build_model_and_opts(RUN_CFG)
domain_datasets = load_domain_datasets(RUN_CFG, n_train=N_PRETRAIN_STEPS + 10)

all_batches = [b for batches in domain_datasets.values() for b in batches]
random.shuffle(all_batches)

# Resume from latest Stage 0 checkpoint if available
STAGE = 0
_stage_ckpts = sorted(CKPT_DIR.glob('ckpt_stage0_*.pt'), key=lambda p: p.stat().st_mtime)
_resume_step = 0
if _stage_ckpts:
    _resume_step, _ = load_checkpoint(_stage_ckpts[-1], model, opts, si, schedulers)
    print(f'[Stage 0] Resuming from step {_resume_step}')

wandb.init(project='cfln', name='stage0_lm_warmup', config=cfg_to_dict(RUN_CFG), mode='disabled')

loss_hist, probes = training_loop(
    model, opts, si, schedulers, all_batches, RUN_CFG,
    stage='stage0', start_step=_resume_step, n_steps=N_PRETRAIN_STEPS,
    doc_ctx=doc_ctx,
)

val_result = evaluate(model, all_batches[:50], RUN_CFG)
print(f'Stage 0 done. val_ce={val_result["val_ce"]:.4f}  ppl={val_result["val_ppl"]:.1f}')

save_checkpoint(model, opts, si, schedulers, N_PRETRAIN_STEPS, 'stage0',
                CKPT_DIR / 'ckpt_stage0_final.pt')
# upload_checkpoint(CKPT_DIR/'ckpt_stage0_final.pt', step=N_PRETRAIN_STEPS)

# SI snapshot after Stage 0
si.save_task_snapshot(si._get_named_params(model))

## 11. Stage 1 â€” PSC self-supervised training

`L_PSC = L_LM + Î±Â·L_improve + Î²(U_epi)Â·L_economy + Î³Â·L_predictive`

Stop signal: `L_improve / L_LM < 0.05` â€” thinking chain working.
Expected time: ~1.5h for 5K steps on T4.

In [ ]:
from cfln.training.train_step import psc_train_step

N_PSC_STEPS = 50   # set to ~5000 for a real run

print(f'=== STAGE 1 PSC TRAINING  ({N_PSC_STEPS} steps) ===')

# Resume from latest Stage 1 checkpoint if available
STAGE = 1
_stage_ckpts = sorted(CKPT_DIR.glob('ckpt_stage1_*.pt'), key=lambda p: p.stat().st_mtime)
_resume_step = 0
if _stage_ckpts:
    _resume_step, _ = load_checkpoint(_stage_ckpts[-1], model, opts, si, schedulers)
    print(f'[Stage 1] Resuming from step {_resume_step}')

wandb.init(project='cfln', name='stage1_psc', config=cfg_to_dict(CFG_PSC), mode='disabled')

def psc_fn(batch, step):
    return psc_train_step(
        batch, model, psc_loss_fn, opts, si,
        phase='psc', step=step, total_steps=N_PSC_STEPS,
        cfg=cfg_to_dict(CFG_PSC), doc_ctx=doc_ctx,
    )

psc_loss_hist, psc_probes = training_loop(
    model, opts, si, schedulers, all_batches, CFG_PSC,
    stage='stage1', start_step=_resume_step, n_steps=N_PSC_STEPS,
    train_fn=psc_fn, doc_ctx=doc_ctx,
)

print(f'PSC done. Final loss: {psc_loss_hist[-1]:.4f}')
save_checkpoint(model, opts, si, schedulers, N_PSC_STEPS, 'stage1',
                CKPT_DIR / 'ckpt_stage1_final.pt')

si.save_task_snapshot(si._get_named_params(model))

## 12. Stage 2 â€” RPP-STaR trace generation

Pure inference â€” no gradients. Generates training traces for SFT.
Target acceptance rate: >70%. If <50%, re-run Stage 1 with more steps.
Expected time: ~45 min.

In [ ]:
try:
    from cfln.training.rpp import star_generate_traces_rpp
    HAS_RPP = True
except ImportError:
    HAS_RPP = False
    print('rpp module not yet implemented â€” skipping trace generation')

TRACES_PATH = CKPT_DIR / 'rpp_traces.pt'

if HAS_RPP:
    print('=== STAGE 2 RPP-STaR TRACE GENERATION ===')
    if TRACES_PATH.exists():
        print(f'Traces already exist at {TRACES_PATH} â€” loading.')
        traces = torch.load(TRACES_PATH)
        accepted = [t for t in traces if t.get('accepted', False)]
        print(f'Loaded {len(accepted)} accepted traces')
    else:
        model.eval()
        model.reset_for_inference()
        prompts = [b['input_ids'][:, :64] for b in all_batches[:1000]]
        traces = star_generate_traces_rpp(
            model, prompts,
            n_traces_target=5_000,
            n_think=8, cfg=cfg_to_dict(CFG_PSC)
        )
        accepted = [t for t in traces if t.get('accepted', False)]
        print(f'Acceptance rate: {len(accepted)/max(len(traces),1):.1%}  (target >70%)')
        torch.save(traces, TRACES_PATH)
        print(f'Saved {len(traces)} traces to {TRACES_PATH}')
        model.train()
else:
    # Create placeholder accepted traces from batches for pipeline testing
    accepted = [{'input_ids': b['input_ids'], 'accepted': True}
                for b in all_batches[:100]]

## 13. Stage 3 â€” SFT on RPP traces

Teaches WHAT to write in thinking tokens.
Expected time: ~45 min for 2K steps.

In [ ]:
try:
    from cfln.training.train_step import sft_train_step_ctp
    HAS_SFT = True
except ImportError:
    HAS_SFT = False
    print('sft_train_step_ctp not yet implemented â€” skipping Stage 3')

if HAS_SFT and accepted:
    N_SFT_STEPS = 50   # set to ~2000 for a real run
    print(f'=== STAGE 3 SFT  ({N_SFT_STEPS} steps) ===')

    # Resume from latest Stage 3 checkpoint if available
    STAGE = 3
    _stage_ckpts = sorted(CKPT_DIR.glob('ckpt_stage3_*.pt'), key=lambda p: p.stat().st_mtime)
    _resume_step = 0
    if _stage_ckpts:
        _resume_step, _ = load_checkpoint(_stage_ckpts[-1], model, opts, si, schedulers)
        print(f'[Stage 3] Resuming from step {_resume_step}')
    elif TRACES_PATH.exists():
        all_traces = torch.load(TRACES_PATH)
        accepted = [t for t in all_traces if t.get('accepted')]

    CFG_S3 = make_ablation_cfg(stage=3)

    def sft_fn(batch, step):
        return sft_train_step_ctp(batch, model, opts, si, cfg_to_dict(CFG_S3))

    sft_loss_hist, _ = training_loop(
        model, opts, si, schedulers, accepted, CFG_S3,
        stage='stage3', start_step=_resume_step, n_steps=N_SFT_STEPS,
        train_fn=sft_fn, doc_ctx=doc_ctx,
    )
    print(f'SFT done. Final loss: {sft_loss_hist[-1]:.4f}')
    save_checkpoint(model, opts, si, schedulers, N_SFT_STEPS, 'stage3',
                    CKPT_DIR / 'ckpt_stage3_final.pt')

## 14. Stage 4 â€” GRPO fine-tuning

Reinforces reasoning quality via intrinsic perplexity-reduction reward.
**CRITICAL**: `ref_model` must be a frozen `deepcopy` of Stage 3 model.
Expected time: ~2h for 1K steps (G=8 rollouts is expensive).

In [ ]:
try:
    from cfln.training.train_step import grpo_train_step
    HAS_GRPO = True
except ImportError:
    HAS_GRPO = False
    print('grpo_train_step not yet implemented â€” skipping Stage 4')

if HAS_GRPO:
    N_GRPO_STEPS = 20   # set to ~1000 for a real run
    print(f'=== STAGE 4 GRPO  ({N_GRPO_STEPS} steps) ===')

    # Resume from latest Stage 4 checkpoint if available
    STAGE = 4
    _stage_ckpts = sorted(CKPT_DIR.glob('ckpt_stage4_*.pt'), key=lambda p: p.stat().st_mtime)
    _resume_step = 0
    if _stage_ckpts:
        _resume_step, _ = load_checkpoint(_stage_ckpts[-1], model, opts, si, schedulers)
        print(f'[Stage 4] Resuming from step {_resume_step}')

    # CRITICAL: frozen reference copy before training begins
    ref_model = copy.deepcopy(model)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    def grpo_fn(batch, step):
        return grpo_train_step(batch, model, ref_model, opts, cfg_to_dict(CFG_GRPO))

    grpo_loss_hist, _ = training_loop(
        model, opts, si, schedulers, all_batches, CFG_GRPO,
        stage='stage4', start_step=_resume_step, n_steps=N_GRPO_STEPS,
        train_fn=grpo_fn, doc_ctx=doc_ctx,
    )
    print(f'GRPO done. Final loss: {grpo_loss_hist[-1]:.4f}')
    save_checkpoint(model, opts, si, schedulers, N_GRPO_STEPS, 'stage4',
                    CKPT_DIR / 'ckpt_stage4_final.pt')
    del ref_model
    gc.collect()

## 15. Three-domain CL protocol

Spec Â§5 Phase 3:

| Phase | Domain | Steps | SI action |
|---|---|---|---|
| 1 | LANG | 5K | SI snapshot end |
| 2 | CODE | 5K | SI snapshot end; eval LANG |
| 3 | SCIENCE | 5K | eval LANG + CODE |
| Recovery | LANG | 2K | eval LANG recovery |

Pass criterion: BWT < +2.0 PPL; no collapse (>3Ã-- initial PPL).

In [ ]:
from cfln.training.curriculum import CurriculumSampler

DOMAIN_STEPS   = {'LANG': 100, 'CODE': 100, 'SCIENCE': 100}  # set to 5000 for real run
RECOVERY_STEPS = 40  # set to 2000 for real run

cl_datasets = {
    'LANG':    domain_datasets.get('LANG', []),
    'CODE':    domain_datasets.get('CODE', []),
    'SCIENCE': domain_datasets.get('SCIENCE', []),
}

# Measure initial PPL before CL training
ppl_init = {}
for domain, batches in cl_datasets.items():
    ppl_init[domain] = evaluate(model, batches[:20], RUN_CFG)['val_ce']
    print(f'  Initial CE ({domain}): {ppl_init[domain]:.4f}')


def run_cl_phase(domain, n_steps, label):
    sampler = CurriculumSampler({domain: cl_datasets[domain]},
                                 steps_per_domain=n_steps)
    for step, dom, batch in sampler.generate(n_steps):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        info = train_step_v605(
            batch, model, opts, si,
            phase='pretrain', step=step, total_steps=n_steps,
            cfg=cfg_to_dict(RUN_CFG), doc_ctx=doc_ctx,
        )
        if step % LOG_EVERY == 0:
            print(f'  [{label}] step {step}  loss={info.get("L_task", float("nan")):.4f}')


print('=== CL Phase 1: LANG ===')
run_cl_phase('LANG', DOMAIN_STEPS['LANG'], 'LANG')
si.save_task_snapshot(si._get_named_params(model))
ppl_after_lang = evaluate(model, cl_datasets['LANG'], RUN_CFG)['val_ce']
print(f'Post-LANG CE: {ppl_after_lang:.4f}')

print('=== CL Phase 2: CODE ===')
run_cl_phase('CODE', DOMAIN_STEPS['CODE'], 'CODE')
si.save_task_snapshot(si._get_named_params(model))
ppl_lang_after_code = evaluate(model, cl_datasets['LANG'], RUN_CFG)['val_ce']
ppl_code_after_code = evaluate(model, cl_datasets['CODE'], RUN_CFG)['val_ce']
print(f'Post-CODE CE  LANG={ppl_lang_after_code:.4f}  CODE={ppl_code_after_code:.4f}')

print('=== CL Phase 3: SCIENCE ===')
run_cl_phase('SCIENCE', DOMAIN_STEPS['SCIENCE'], 'SCIENCE')
ppl_lang_after_sci = evaluate(model, cl_datasets['LANG'],    RUN_CFG)['val_ce']
ppl_code_after_sci = evaluate(model, cl_datasets['CODE'],    RUN_CFG)['val_ce']
ppl_sci_after_sci  = evaluate(model, cl_datasets['SCIENCE'], RUN_CFG)['val_ce']
print(f'Post-SCIENCE CE  LANG={ppl_lang_after_sci:.4f}  CODE={ppl_code_after_sci:.4f}  SCIENCE={ppl_sci_after_sci:.4f}')

print('=== CL Recovery: LANG ===')
run_cl_phase('LANG', RECOVERY_STEPS, 'RECOVERY')
ppl_lang_recovery = evaluate(model, cl_datasets['LANG'], RUN_CFG)['val_ce']
print(f'Post-recovery CE (LANG): {ppl_lang_recovery:.4f}')

## 16. BWT evaluation

Backward Transfer measures forgetting. Pass criterion: BWT_A, BWT_B < +2.0 CE;
no collapse (>3Ã-- initial CE).

In [ ]:
if not all(v in dir() for v in ['ppl_after_lang', 'ppl_lang_after_code',
                                  'ppl_code_after_code', 'ppl_code_after_sci',
                                  'ppl_lang_after_sci', 'ppl_lang_recovery']):
    print('BWT: CL variables not available â€” re-run the CL phase cell first')
else:
    print('=== BWT EVALUATION ===')
    
    BWT_A = ppl_lang_after_code - ppl_after_lang
    BWT_B = ppl_code_after_sci  - ppl_code_after_code
    
    print(f'  CE(LANG) after LANG training:     {ppl_after_lang:.4f}')
    print(f'  CE(LANG) after CODE training:     {ppl_lang_after_code:.4f}')
    print(f'  CE(LANG) after SCIENCE training:  {ppl_lang_after_sci:.4f}')
    print(f'  CE(LANG) after recovery:          {ppl_lang_recovery:.4f}')
    print()
    print(f'  CE(CODE) after CODE training:     {ppl_code_after_code:.4f}')
    print(f'  CE(CODE) after SCIENCE training:  {ppl_code_after_sci:.4f}')
    print()
    print(f'  BWT_A (LANG forgetting): {BWT_A:+.4f}  threshold=+2.0')
    print(f'  BWT_B (CODE forgetting): {BWT_B:+.4f}  threshold=+2.0')
    
    collapse_A = ppl_lang_after_sci > 3.0 * ppl_init.get('LANG', ppl_after_lang)
    collapse_B = ppl_code_after_sci > 3.0 * ppl_init.get('CODE', ppl_code_after_code)
    pass_A = BWT_A < 2.0 and not collapse_A
    pass_B = BWT_B < 2.0 and not collapse_B
    
    print(f'\n  PASS A: {pass_A}  (BWT_A < 2.0 and no collapse)')
    print(f'  PASS B: {pass_B}  (BWT_B < 2.0 and no collapse)')
    
    bwt_results = {
        'BWT_A': BWT_A, 'BWT_B': BWT_B,
        'pass_A': pass_A, 'pass_B': pass_B,
        'collapse_A': collapse_A, 'collapse_B': collapse_B,
    }
    with open(OUT_DIR / 'bwt_results.json', 'w') as f:
        json.dump(bwt_results, f, indent=2)
    print('BWT results saved to', OUT_DIR / 'bwt_results.json')

## 17. NeedleInHaystack tiered evaluation

Tests retrieval at five distances: `C//2, CÃ--4, CÃ--16, CÃ--64, CÃ--128`.

In [ ]:
print('=== NEEDLE-IN-HAYSTACK EVALUATION ===')
C = RUN_CFG.C_chunk
distances = [C // 2, C * 4, C * 16, C * 64, C * 128]
V = RUN_CFG.vocab_size
needle_token = 3

nih_results = {}
model.eval()

for dist in distances:
    seq_len = min(dist + 2, RUN_CFG.T)
    seq = [needle_token] + [random.randint(4, V - 1) for _ in range(seq_len - 1)]
    ids = torch.tensor([seq], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        logits, _, _ = model(ids, training=False)
        probs = F.softmax(logits[0], dim=-1)
        needle_prob = float(probs[-1, needle_token].mean())
    nih_results[dist] = needle_prob
    print(f'  dist={dist:6d}  P(needle)={needle_prob:.4f}')

model.train()
with open(OUT_DIR / 'nih_results.json', 'w') as f:
    json.dump({str(k): v for k, v in nih_results.items()}, f, indent=2)
print('NIH results saved.')

## 18. Ablation series

All ablations run from the same Stage 0 checkpoint. Optimizer state reset for fair comparison.

| ID | Tests | Config change |
|---|---|---|
| A85 | PSC â€” L_improve only | psc_beta_max=0, psc_gamma=0 |
| A86 | PSC â€” L_improve + L_economy | psc_gamma=0 |
| A87 | Full PSC | = Stage 1 |
| A88 | PSC + SFT (no GRPO) | = Stage 3 final |
| A89 | Full pipeline | = Stage 4 final |
| A90 | Cold STaR + GRPO (no PSC) | skip Stages 1+2 |
| A93 | 2-tier vs 3-tier CFLN | restore n_g=64 |

In [ ]:
def ablation_reset_optimizer(model, cfg):
    """Fresh optimizer state â€” fair comparison across ablations."""
    cfg_dict = cfg_to_dict(cfg) if not isinstance(cfg, dict) else cfg
    new_opts = build_optimizers_v605(model, cfg_dict)
    warmup = cfg_dict.get('warmup_steps', 500)
    new_scheds = {
        'sched_g': torch.optim.lr_scheduler.LambdaLR(
            new_opts[2], lambda s: min(1.0, (s+1)/warmup)),
        'sched_u': torch.optim.lr_scheduler.LambdaLR(
            new_opts[3], lambda s: min(1.0, (s+1)/warmup)),
    }
    return new_opts, new_scheds


def run_ablation(label, cfg_overrides, n_steps=20, train_fn_builder=None):
    """Run one ablation from Stage 0 checkpoint."""
    # Load Stage 0 base â€” keep SI, reset optimizer
    abl_cfg = make_ablation_cfg(**cfg_overrides)
    abl_opts, abl_scheds = ablation_reset_optimizer(model, abl_cfg)
    # load_checkpoint(INPUT/'cfln-checkpoints/ckpt_stage0_final.pt',
    #                 model, abl_opts, si, abl_scheds)

    fn = train_fn_builder(abl_opts) if train_fn_builder else None
    loss_hist, probes = training_loop(
        model, abl_opts, si, abl_scheds, all_batches, abl_cfg,
        stage=label, start_step=0, n_steps=n_steps,
        train_fn=fn, doc_ctx=doc_ctx,
    )
    final_ce = sum(loss_hist[-5:]) / 5
    print(f'  [{label}] mean CE (last 5): {final_ce:.4f}')
    gc.collect()
    return final_ce


N_ABL = 20  # set to ~5000 for real ablations

print('=== ABLATION A93: 2-tier vs 3-tier ===')
loss_2tier = run_ablation('A93_2tier', {}, n_steps=N_ABL)

print('=== ABLATION A85: PSC L_improve only ===')
loss_a85 = run_ablation('A85', {'psc_beta_max': 0.0, 'psc_gamma': 0.0}, n_steps=N_ABL)

print('=== ABLATION A86: PSC L_improve + L_economy ===')
loss_a86 = run_ablation('A86', {'psc_gamma': 0.0}, n_steps=N_ABL)

print(f'\nA93 2-tier CE: {loss_2tier:.4f}')
print(f'A85 CE: {loss_a85:.4f}  A86 CE: {loss_a86:.4f}')

In [ ]:
# â”€â”€ A90: Ablation â€” disable phase binding â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== ABLATION A90: No Phase Binding ===')
import copy as _copy
_model_no_bind = _copy.deepcopy(model)
with torch.no_grad():
    for _layer in _model_no_bind.layers:
        _layer.bank.log_lam_bind.fill_(-30.0)   # exp(-30) â‰ˆ 0

_a90_losses = []
_a90_opts, _a90_scheds = ablation_reset_optimizer(_model_no_bind, ABLATION)
_a90_si = type(si)(c_SI=si.c_SI)
for _step, _batch in zip(range(N_ABL), all_batches):
    _batch = {k: v.to(DEVICE) for k, v in _batch.items()}
    with torch.no_grad():
        _info = train_step_v605(
            _batch, _model_no_bind, _a90_opts, _a90_si,
            phase='pretrain', step=_step, total_steps=N_ABL,
            cfg=cfg_to_dict(ABLATION), doc_ctx=None,
        )
    _a90_losses.append(_info.get('L_task', float('nan')))

_a90_mean = sum(_a90_losses) / max(len(_a90_losses), 1)
_baseline_ce = sum(loss_hist[-5:]) / 5 if 'loss_hist' in dir() and loss_hist else float('nan')
print(f'[A90 No-PhaseBinding] mean L_task = {_a90_mean:.4f}')
print(f'[Baseline Stage 0]    mean L_task = {_baseline_ce:.4f}')
del _model_no_bind, _a90_opts, _a90_si
gc.collect()

## 19. Generation demo â€” DCG+

In [ ]:
from cfln.inference.dcg import generate_cfln_dcg_plus

prompt_tokens = [1, 2, 5, 10, 20]
prompt = torch.tensor([prompt_tokens], dtype=torch.long, device=DEVICE)

print('=== DCG+ GENERATION ===')
model.eval()
dcg_out = generate_cfln_dcg_plus(
    model, prompt,
    max_new_tokens=20,
    temperature=1.0,
)
print('  Output tokens:', dcg_out.tolist())
model.train()

## Summary

Results written to `/kaggle/working/cfln_output/`:

| File | Contents |
|---|---|
| `checkpoints/ckpt_stage0_final.pt` | Stage 0 model + SI |
| `checkpoints/ckpt_stage1_final.pt` | Stage 1 PSC |
| `checkpoints/ckpt_stage3_final.pt` | Stage 3 SFT |
| `checkpoints/ckpt_stage4_final.pt` | Stage 4 GRPO |
| `bwt_results.json` | BWT_A, BWT_B, pass/fail |
| `nih_results.json` | NeedleInHaystack retrieval |

**Pass criteria (spec Â§5)**
- `BWT_A < +2.0 CE` â€” no LANG forgetting after CODE training
- `BWT_B < +2.0 CE` â€” no CODE forgetting after SCIENCE training
- No collapse (`> 3Ã--` initial CE)

**To run the full protocol**, set:
```python
DOMAIN_STEPS   = {'LANG': 5000, 'CODE': 5000, 'SCIENCE': 5000}
RECOVERY_STEPS = 2000
N_PRETRAIN_STEPS = 10000
N_PSC_STEPS      = 5000
RUN_CFG = FULL
```
and enable a T4 Ã-- 2 accelerator.

**Session plan** (3 sessions to trained model):
```
Session 1 (internet ON, 12h): build tokenizer â†’ Stage 0 â†’ Stage 1 â†’ upload checkpoints
Session 2 (internet OFF, 9h): Stage 2 traces â†’ Stage 3 SFT â†’ Stage 4 GRPO â†’ A85/A86
Session 3 (internet OFF, 9h): A90 cold STaR â†’ A93 3-tier â†’ comparative eval
```

**OQ targets** (logged automatically via `OQProbes`):
- `u_epi_ce_corr > 0.4` â€” U_epi calibration (OQ-v598-2)
- `escape_rate` trending down â€” rule cache working (OQ-v598-4)
- `n_active_units` grows then stabilises â€” unit lifecycle healthy